# One-Go Colab Training Pipeline: Hybrid CNN + ViT

This notebook runs the full pipeline in one flow:
1. Environment setup
2. Clone/load project
3. Extract raw RDD zip files from Google Drive
4. Preprocess dataset (70/15/15)
5. Train hybrid detector (staged unfreezing, fp16)
6. Evaluate (mAP/IoU/F1), export ONNX
7. Save artifacts to project and Google Drive

## Before Run-All

- In Colab: Runtime > Change runtime type > GPU
- Put your raw RDD zip files in: `MyDrive/rdd2022_raw_zips`
- Then run: Runtime > Run all

In [ ]:
!pip -q install timm==1.0.9 torchmetrics==1.4.0 pycocotools opencv-python-headless==4.10.0.84

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import math
import os
import random
import shutil
import subprocess
import sys
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchmetrics.detection.mean_ap import MeanAveragePrecision

In [ ]:
REPO_URL = 'https://github.com/LakshmananVS12/projectfy.git'
REPO_DIR = Path('/content/projectfy')

if not (REPO_DIR / '.git').exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already present at', REPO_DIR)

PROJECT_ROOT = REPO_DIR
AI_SERVICE_ROOT = PROJECT_ROOT / 'ai-service'
DATA_RAW_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'RDD2022'
DATA_PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'rdd2022'
TRAINING_ARTIFACTS = PROJECT_ROOT / 'training' / 'artifacts'
WEIGHTS_DIR = PROJECT_ROOT / 'ai-service' / 'weights'

TRAINING_ARTIFACTS.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

if str(AI_SERVICE_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_SERVICE_ROOT))

required_files = [
    PROJECT_ROOT / 'training' / 'train_hybrid_model_one_go.ipynb',
    AI_SERVICE_ROOT / 'model' / 'hybrid_detector.py',
    AI_SERVICE_ROOT / 'model' / 'losses.py',
    AI_SERVICE_ROOT / 'model' / 'postprocess.py',
    AI_SERVICE_ROOT / 'data' / 'preprocess_rdd2022.py',
]

missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required project files: ' + str(missing))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('AI_SERVICE_ROOT =', AI_SERVICE_ROOT)

In [ ]:
DRIVE_ZIP_DIR = Path('/content/drive/MyDrive/rdd2022_raw_zips')
DATA_RAW_ROOT.mkdir(parents=True, exist_ok=True)

if not DRIVE_ZIP_DIR.exists():
    raise FileNotFoundError(f'Drive zip directory not found: {DRIVE_ZIP_DIR}')

zip_files = sorted(DRIVE_ZIP_DIR.rglob('*.zip'))
if not zip_files:
    raise FileNotFoundError(f'No zip files found under: {DRIVE_ZIP_DIR}')

print('Found zip files:', len(zip_files))
for zf in zip_files:
    print(' -', zf)

for zf in zip_files:
    with zipfile.ZipFile(zf, 'r') as archive:
        archive.extractall(DATA_RAW_ROOT)

xml_count = len(list(DATA_RAW_ROOT.rglob('*.xml')))
img_count = (
    len(list(DATA_RAW_ROOT.rglob('*.jpg')))
    + len(list(DATA_RAW_ROOT.rglob('*.jpeg')))
    + len(list(DATA_RAW_ROOT.rglob('*.png')))
)

print('Raw extraction complete.')
print('XML count:', xml_count)
print('Image count:', img_count)

if xml_count == 0 or img_count == 0:
    raise RuntimeError('Extraction did not produce both XML annotations and images. Check zip contents.')

In [ ]:
preprocess_script = AI_SERVICE_ROOT / 'data' / 'preprocess_rdd2022.py'
cmd = [
    sys.executable,
    str(preprocess_script),
    '--raw-dir', str(DATA_RAW_ROOT),
    '--output-dir', str(DATA_PROCESSED_ROOT),
    '--image-size', '640',
    '--seed', '42',
    '--min-ravelling-samples', '300',
]

print('Running preprocessing:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)

expected = [
    DATA_PROCESSED_ROOT / 'annotations_train.json',
    DATA_PROCESSED_ROOT / 'annotations_val.json',
    DATA_PROCESSED_ROOT / 'annotations_test.json',
    DATA_PROCESSED_ROOT / 'preprocess_report.json',
]
for p in expected:
    print(p.name, '->', p.exists())

if not all(p.exists() for p in expected):
    raise RuntimeError('Preprocessing outputs are incomplete.')

report = json.loads((DATA_PROCESSED_ROOT / 'preprocess_report.json').read_text(encoding='utf-8'))
print('Active classes:', report.get('active_classes'))
print('Dropped classes:', report.get('dropped_classes'))
print('Class counts:', report.get('class_counts'))
for note in report.get('notes', []):
    print('-', note)

In [ ]:
from model.hybrid_detector import HybridDetectorConfig, build_hybrid_model
from model.losses import FCOSLoss
from model.postprocess import decode_detections

@dataclass
class TrainConfig:
    image_size: int = 640
    batch_size: int = 8
    epochs_stage1: int = 2
    epochs_stage2: int = 8
    learning_rate_stage1: float = 3e-4
    learning_rate_stage2: float = 1e-4
    weight_decay: float = 1e-4
    num_workers: int = 2
    score_threshold_eval: float = 0.25
    seed: int = 42
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = TrainConfig()
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)

if cfg.device == 'cuda':
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cudnn.benchmark = True

print(cfg)

In [ ]:
class CocoDamageDataset(Dataset):
    def __init__(self, images_dir: Path, annotations_file: Path, train: bool):
        self.images_dir = images_dir
        self.train = train

        if not annotations_file.exists():
            raise FileNotFoundError(f'Annotation file missing: {annotations_file}')

        data = json.loads(annotations_file.read_text(encoding='utf-8'))
        self.images = data['images']
        self.categories = data['categories']

        by_image = {}
        for ann in data['annotations']:
            by_image.setdefault(ann['image_id'], []).append(ann)
        self.by_image = by_image

    def __len__(self):
        return len(self.images)

    def _augment(self, image: np.ndarray, boxes: torch.Tensor):
        if not self.train:
            return image, boxes

        if boxes.shape[0] > 0 and random.random() < 0.5:
            image = image[:, ::-1, :].copy()
            width = image.shape[1]
            x1 = boxes[:, 0].clone()
            x2 = boxes[:, 2].clone()
            boxes[:, 0] = width - x2
            boxes[:, 2] = width - x1

        if random.random() < 0.2:
            gain = random.uniform(0.85, 1.15)
            image = np.clip(image.astype(np.float32) * gain, 0, 255).astype(np.uint8)

        return image, boxes

    def __getitem__(self, idx):
        item = self.images[idx]
        image_path = self.images_dir / item['file_name']
        image = cv2.imread(str(image_path))
        if image is None:
            raise FileNotFoundError(f'Image file missing/unreadable: {image_path}')

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        anns = self.by_image.get(item['id'], [])
        boxes = []
        labels = []
        for ann in anns:
            x, y, w, h = ann['bbox']
            boxes.append([x, y, x + w, y + h])
            labels.append(ann['category_id'])

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.float32)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.float32)

        image, boxes = self._augment(image, boxes)
        image = np.ascontiguousarray(image)
        image_tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0

        if boxes.shape[0] == 0:
            target = torch.zeros((0, 5), dtype=torch.float32)
        else:
            target = torch.cat([boxes, labels.unsqueeze(1)], dim=1)

        return image_tensor, target


def collate_fn(batch):
    images = torch.stack([x[0] for x in batch], dim=0)
    targets = [x[1] for x in batch]
    return images, targets

In [ ]:
train_ds = CocoDamageDataset(
    images_dir=DATA_PROCESSED_ROOT / 'images' / 'train',
    annotations_file=DATA_PROCESSED_ROOT / 'annotations_train.json',
    train=True,
)

val_ds = CocoDamageDataset(
    images_dir=DATA_PROCESSED_ROOT / 'images' / 'val',
    annotations_file=DATA_PROCESSED_ROOT / 'annotations_val.json',
    train=False,
)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
    collate_fn=collate_fn,
)

print('Train images:', len(train_ds), 'Val images:', len(val_ds))

In [ ]:
model_cfg = HybridDetectorConfig(
    num_classes=4,
    class_names=('pothole', 'linear_crack', 'alligator_crack', 'edge_break'),
    cnn_variant='resnet34',
    cnn_output_stage='layer3',
    vit_variant='deit_small_patch16_224',
    pretrained_backbones=True,
)

model = build_hybrid_model(model_cfg).to(cfg.device)
criterion = FCOSLoss(num_classes=model_cfg.num_classes, stride=model.stride)
scaler = torch.cuda.amp.GradScaler(enabled=(cfg.device == 'cuda'))

def build_optimizer(lr: float):
    params = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(params, lr=lr, weight_decay=cfg.weight_decay)

print('Model stride:', model.stride)

In [ ]:
def to_metric_format(decoded, targets):
    preds = []
    refs = []

    for pred_det, tgt in zip(decoded, targets):
        if len(pred_det) == 0:
            preds.append({
                'boxes': torch.zeros((0, 4), dtype=torch.float32),
                'scores': torch.zeros((0,), dtype=torch.float32),
                'labels': torch.zeros((0,), dtype=torch.int64),
            })
        else:
            preds.append({
                'boxes': torch.tensor([d['bbox'] for d in pred_det], dtype=torch.float32),
                'scores': torch.tensor([d['confidence'] for d in pred_det], dtype=torch.float32),
                'labels': torch.tensor([d['class_id'] for d in pred_det], dtype=torch.int64),
            })

        if tgt.shape[0] == 0:
            refs.append({
                'boxes': torch.zeros((0, 4), dtype=torch.float32),
                'labels': torch.zeros((0,), dtype=torch.int64),
            })
        else:
            refs.append({
                'boxes': tgt[:, :4].cpu().float(),
                'labels': tgt[:, 4].cpu().long(),
            })

    return preds, refs


def iou_xyxy(a: torch.Tensor, b: torch.Tensor) -> float:
    x1 = max(float(a[0]), float(b[0]))
    y1 = max(float(a[1]), float(b[1]))
    x2 = min(float(a[2]), float(b[2]))
    y2 = min(float(a[3]), float(b[3]))
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = max(0.0, float(a[2] - a[0])) * max(0.0, float(a[3] - a[1]))
    area_b = max(0.0, float(b[2] - b[0])) * max(0.0, float(b[3] - b[1]))
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def batch_iou_f1(decoded, targets, iou_thresh=0.5):
    tp, fp, fn = 0, 0, 0
    iou_sum = 0.0
    matches = 0

    for pred_det, tgt in zip(decoded, targets):
        gt = tgt.cpu()
        used_gt = set()

        for pred in pred_det:
            pbox = torch.tensor(pred['bbox'], dtype=torch.float32)
            plabel = int(pred['class_id'])

            best_iou = 0.0
            best_idx = -1
            for gi in range(gt.shape[0]):
                if gi in used_gt:
                    continue
                if int(gt[gi, 4].item()) != plabel:
                    continue
                curr_iou = iou_xyxy(pbox, gt[gi, :4])
                if curr_iou > best_iou:
                    best_iou = curr_iou
                    best_idx = gi

            if best_idx >= 0 and best_iou >= iou_thresh:
                tp += 1
                used_gt.add(best_idx)
                iou_sum += best_iou
                matches += 1
            else:
                fp += 1

        fn += max(0, gt.shape[0] - len(used_gt))

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = (2 * precision * recall) / (precision + recall + 1e-8)
    mean_iou = iou_sum / (matches + 1e-8)
    return float(mean_iou), float(f1)


def run_eval(model, loader):
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    iou_vals = []
    f1_vals = []

    model.eval()
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(cfg.device, non_blocking=True)
            outputs = model(images)

            decoded = decode_detections(
                cls_logits=outputs['cls_logits'],
                bbox_reg=outputs['bbox_reg'],
                centerness=outputs['centerness'],
                stride=model.stride,
                class_names=model_cfg.class_names,
                score_threshold=cfg.score_threshold_eval,
                nms_iou_threshold=0.5,
                top_k=200,
                image_hw=(cfg.image_size, cfg.image_size),
            )

            preds, refs = to_metric_format(decoded, targets)
            metric.update(preds, refs)

            mean_iou, f1 = batch_iou_f1(decoded, targets)
            iou_vals.append(mean_iou)
            f1_vals.append(f1)

    map_scores = metric.compute()
    return {
        'mAP': float(map_scores.get('map', torch.tensor(0.0)).item()),
        'mAP@50': float(map_scores.get('map_50', torch.tensor(0.0)).item()),
        'mean_iou': float(np.mean(iou_vals) if iou_vals else 0.0),
        'f1': float(np.mean(f1_vals) if f1_vals else 0.0),
    }

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    running = 0.0

    autocast_enabled = cfg.device == 'cuda'
    autocast_device = 'cuda' if autocast_enabled else 'cpu'

    for images, targets in loader:
        images = images.to(cfg.device, non_blocking=True)
        targets = [t.to(cfg.device) for t in targets]

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=autocast_device, enabled=autocast_enabled, dtype=torch.float16):
            outputs = model(images)
            losses = criterion(outputs['cls_logits'], outputs['bbox_reg'], outputs['centerness'], targets)
            loss = losses['loss_total']

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running += float(loss.item())

    return running / max(1, len(loader))


history = []
best_map50 = -1.0
best_ckpt = TRAINING_ARTIFACTS / 'hybrid_detector_best.pt'

# Stage 1: freeze pretrained backbones, train fusion + detection head from scratch.
model.freeze_backbones()
optimizer = build_optimizer(cfg.learning_rate_stage1)

for epoch in range(cfg.epochs_stage1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    metrics = run_eval(model, val_loader)
    row = {'stage': 1, 'epoch': epoch + 1, 'train_loss': train_loss, **metrics}
    history.append(row)
    print(row)

    if metrics['mAP@50'] > best_map50:
        best_map50 = metrics['mAP@50']
        torch.save({'model_state': model.state_dict(), 'config': model_cfg.__dict__, 'metrics': row}, best_ckpt)

# Stage 2: unfreeze final backbone blocks for task adaptation.
model.unfreeze_vit_last_blocks(num_blocks=2)
model.unfreeze_cnn_last_stage()
optimizer = build_optimizer(cfg.learning_rate_stage2)

for epoch in range(cfg.epochs_stage2):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    metrics = run_eval(model, val_loader)
    row = {'stage': 2, 'epoch': epoch + 1, 'train_loss': train_loss, **metrics}
    history.append(row)
    print(row)

    if metrics['mAP@50'] > best_map50:
        best_map50 = metrics['mAP@50']
        torch.save({'model_state': model.state_dict(), 'config': model_cfg.__dict__, 'metrics': row}, best_ckpt)

history_path = TRAINING_ARTIFACTS / 'train_history.json'
history_path.write_text(json.dumps(history, indent=2), encoding='utf-8')
print('Best mAP@50:', best_map50)
print('Best checkpoint:', best_ckpt)

In [ ]:
state = torch.load(TRAINING_ARTIFACTS / 'hybrid_detector_best.pt', map_location=cfg.device)
model.load_state_dict(state['model_state'])
model.eval()

class OnnxExportWrapper(nn.Module):
    def __init__(self, detector):
        super().__init__()
        self.detector = detector

    def forward(self, x):
        out = self.detector(x)
        return out['cls_logits'], out['bbox_reg'], out['centerness']

wrapper = OnnxExportWrapper(model).to(cfg.device)
dummy = torch.randn(1, 3, cfg.image_size, cfg.image_size, device=cfg.device)
onnx_path = TRAINING_ARTIFACTS / 'hybrid_detector.onnx'

torch.onnx.export(
    wrapper,
    dummy,
    str(onnx_path),
    input_names=['image'],
    output_names=['cls_logits', 'bbox_reg', 'centerness'],
    dynamic_axes={
        'image': {0: 'batch'},
        'cls_logits': {0: 'batch'},
        'bbox_reg': {0: 'batch'},
        'centerness': {0: 'batch'},
    },
    opset_version=17,
)

print('ONNX exported to', onnx_path)

In [ ]:
best_pt = TRAINING_ARTIFACTS / 'hybrid_detector_best.pt'
onnx_file = TRAINING_ARTIFACTS / 'hybrid_detector.onnx'
history_file = TRAINING_ARTIFACTS / 'train_history.json'

if not best_pt.exists() or not onnx_file.exists():
    raise RuntimeError('Expected training artifacts are missing.')

target_pt = WEIGHTS_DIR / 'hybrid_detector_best.pt'
target_onnx = WEIGHTS_DIR / 'hybrid_detector.onnx'
shutil.copy2(best_pt, target_pt)
shutil.copy2(onnx_file, target_onnx)

drive_out = Path('/content/drive/MyDrive/road-damage-artifacts')
drive_out.mkdir(parents=True, exist_ok=True)

for src in [best_pt, onnx_file, history_file, DATA_PROCESSED_ROOT / 'preprocess_report.json']:
    if src.exists():
        shutil.copy2(src, drive_out / src.name)

print('Saved in project:')
print(' -', target_pt)
print(' -', target_onnx)
print('Saved in Drive:', drive_out)

## Done

After successful run:
- The trained checkpoint is at `ai-service/weights/hybrid_detector_best.pt`
- The ONNX model is at `ai-service/weights/hybrid_detector.onnx`
- Training metrics are in `training/artifacts/train_history.json`

Share the final metric output (mAP, mAP@50, IoU, F1), and continue with service endpoints.